In [ ]:
import os
import datetime as dt

import pandas as pd

CUTOVER = pd.Timestamp(2026, 9, 3, tz="UTC")

Plan:
1. Load the 4 exports (gateway_cu, gateway_db, rmrcloud_cu, rmrcloud_db).
2. Normalize each to a common schema: directory, filename, date_created, status, category, checksum, ref_id, source.
3. Filter every source to `date_created >= CUTOVER` so we're comparing the same window across old and new systems.
4. Build one master `filename` key list and count occurrences per source. (Directory is excluded from the key -- it's a mutable workflow-state field that staff update as files get manually reviewed/moved, so it drifts between a system's own CU list and DB over time. Filename is stable. Directory is kept as a descriptive column only.)
5. Classify each key: matched everywhere, legacy-only, gateway-only, internal mismatch (in one system's CU list but not its DB, or vice versa), duplicated.
6. Spot-check field-level differences (status/category/checksum) for matched keys.

In [ ]:
filenames = os.listdir("lists")

df_dict = {}

for filename in filenames:
    key = filename.split(".")[0]
    df_dict[key] = pd.read_pickle(f"lists/{filename}")

{k: v.shape for k, v in df_dict.items()}

In [ ]:
COLUMN_MAP = {
    "ftp_directory": "directory",
    "ftp_filename": "filename",
    "ftp_file_name": "filename",
    "file_directory": "directory",
    "file_name": "filename",
    "status.status": "status",
    "file_category": "category",
    "file_type": "category",
}


def normalize(df: pd.DataFrame, source: str) -> pd.DataFrame:
    df = df.rename(columns=COLUMN_MAP)
    id_col = "task_id" if "task_id" in df.columns else "id"

    out = pd.DataFrame({
        "directory": df["directory"].astype(str).str.strip().str.lstrip("/"),
        "filename": df["filename"].astype(str).str.strip(),
        "date_created": pd.to_datetime(df["date_created"], utc=True),
        "status": df.get("status"),
        "category": df.get("category"),
        "checksum": df.get("checksum"),
        "ref_id": df[id_col],
    })
    out["source"] = source
    return out


normalized = {key: normalize(df, key) for key, df in df_dict.items()}

for key, df in normalized.items():
    print(key, df.shape, df["date_created"].min(), df["date_created"].max())

In [ ]:
all_df = pd.concat(normalized.values(), ignore_index=True)

windowed_df = all_df[all_df["date_created"] >= CUTOVER].copy()

# `directory` is NOT used in the match key: it's a mutable workflow-state field.
# Staff move files into subfolders after the fact (Completed COBRA, AM Review,
# "Joe Working On", etc), which updates the ClickUp custom field but not the
# DB row (which snapshots the directory at webhook time). Joining rmrcloud_cu
# to rmrcloud_db by task_id shows directory matches only ~5% of the time for
# the *same* task, vs ~98% for filename. So filename is the stable identity;
# directory is kept only as a descriptive column below.
windowed_df["key"] = windowed_df["filename"]

print(f"all rows: {len(all_df)}, rows since cutover ({CUTOVER}): {len(windowed_df)}")
windowed_df["source"].value_counts()

In [ ]:
SOURCES = ["rmrcloud_db", "rmrcloud_cu", "gateway_db", "gateway_cu"]

counts = (
    windowed_df
    .pivot_table(index="key", columns="source", values="ref_id", aggfunc="count", fill_value=0)
    .reindex(columns=SOURCES, fill_value=0)
)

counts.sum()

In [ ]:
present = counts > 0

def classify(row):
    in_legacy_db, in_legacy_cu, in_gw_db, in_gw_cu = row["rmrcloud_db"], row["rmrcloud_cu"], row["gateway_db"], row["gateway_cu"]

    if in_legacy_db != in_legacy_cu:
        return "legacy_internal_mismatch"
    if in_gw_db != in_gw_cu:
        return "gateway_internal_mismatch"
    if in_legacy_db and not in_gw_db:
        return "legacy_only"
    if in_gw_db and not in_legacy_db:
        return "gateway_only"
    return "matched"

report = counts.copy()
report["classification"] = present.apply(classify, axis=1)
report["duplicated"] = (counts > 1).any(axis=1)

report["classification"].value_counts()

In [ ]:
mismatches = report[report["classification"] != "matched"].sort_index()
duplicates = report[report["duplicated"]].sort_index()

# attach a representative directory per key (first seen) purely for context,
# since directory isn't part of the match key
sample_directory = windowed_df.drop_duplicates(subset="key", keep="first").set_index("key")["directory"]
mismatches = mismatches.join(sample_directory)
duplicates = duplicates.join(sample_directory)

print(f"mismatched keys: {len(mismatches)}")
print(f"keys with duplicate rows in at least one source: {len(duplicates)}")

mismatches.to_csv("lists/mismatch_report.csv")
duplicates.to_csv("lists/duplicate_report.csv")

mismatches.head(20)

For keys present in both systems, compare `category` (db) and `checksum` (db) between `rmrcloud_db` and `gateway_db` to confirm the new system isn't just creating *a* record, but the *right* one.

In [ ]:
matched_keys = report[report["classification"] == "matched"].index

def pivot_field(field):
    return (
        windowed_df[windowed_df["source"].isin(["rmrcloud_db", "gateway_db"])]
        .drop_duplicates(subset=["key", "source"], keep="first")
        .pivot(index="key", columns="source", values=field)
        .reindex(columns=["rmrcloud_db", "gateway_db"])
        .reindex(matched_keys)
    )

category_pivot = pivot_field("category")
checksum_pivot = pivot_field("checksum")

# legacy (rmrcloud_db) has no checksum column at all, so only compare checksum
# where the legacy side actually has one -- otherwise every matched row would
# spuriously flag as a mismatch.
checksum_comparable = checksum_pivot["rmrcloud_db"].notna()

is_mismatch = (
    category_pivot["rmrcloud_db"].ne(category_pivot["gateway_db"]) |
    (checksum_comparable & checksum_pivot["rmrcloud_db"].ne(checksum_pivot["gateway_db"]))
)

field_mismatches = pd.DataFrame({
    "category_rmrcloud_db": category_pivot["rmrcloud_db"],
    "category_gateway_db": category_pivot["gateway_db"],
    "checksum_rmrcloud_db": checksum_pivot["rmrcloud_db"],
    "checksum_gateway_db": checksum_pivot["gateway_db"],
})[is_mismatch]

print(f"matched keys with a category/checksum mismatch: {len(field_mismatches)}")
# field_mismatches.head(20)
field_mismatches